## ⚡ GPU Limit Hit? → Use Kaggle (free 30 h/week)

**Kaggle Notebooks** = same free T4/P100 GPU, **no session cut-offs**, 12 h runs.

1. Go to [kaggle.com/code](https://www.kaggle.com/code) → **New Notebook**
2. Settings → **Accelerator: GPU T4 x2** (or P100)
3. Upload `BrainTumorAI_code.zip` via *Add Data → Upload*
4. Copy-paste cells from this notebook — everything works identically
5. Files persist in `/kaggle/working/`

> This notebook is now set to **LITE mode** (7-9 min on T4) to stay within Colab limits.

# 🧠 NeuroScan AI — U-Net GPU Training (Google Colab)
### Run every cell top-to-bottom. All fixes are pre-applied.

| Step | What |
|------|------|
| 1 | GPU check |
| 2 | Mount Google Drive |
| 3 | Upload project code zip |
| 4 | Install packages |
| 5 | Upload MRI dataset (USE-Me Test) |
| 6 | Generate **classical** pseudo masks (no GradCAM) |
| 7 | Train Attention U-Net (~8 min on T4) |
| 8 | Save to Drive + download best_unet.pth |

**Before running:** `Runtime → Change runtime type → T4 GPU → Save`


## Step 1 â€” Verify GPU

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Go to Runtime -> Change runtime type -> T4 GPU")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"CUDA   : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")
print("OK")

## Step 2 â€” Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/NeuroScan_UNet'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive mounted. Outputs -> {DRIVE_DIR}")

## Step 3 â€” Upload Project Code

The zip `BrainTumorAI_code.zip` was already created on your PC at:
`C:\Users\HP\Desktop\PRO(B)\BrainTumorAI\BrainTumorAI_code.zip`

Run the cell below â†’ file picker opens â†’ select that zip.

In [ ]:
from google.colab import files
import zipfile, os, shutil, sys

PROJECT = '/content/BrainTumorAI'
os.makedirs(PROJECT, exist_ok=True)

print("Select BrainTumorAI_code.zip when the picker opens...")
uploaded = files.upload()

for fname in uploaded:
    print(f"Extracting: {fname}")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/tmp/extracted/')
    os.remove(fname)

extracted = '/tmp/extracted'
items    = os.listdir(extracted)
subdirs  = [d for d in items if os.path.isdir(os.path.join(extracted, d))]
topfiles = [f for f in items if os.path.isfile(os.path.join(extracted, f))]
src = os.path.join(extracted, subdirs[0]) if (len(subdirs)==1 and not topfiles) else extracted

def _copytree(s, d):
    os.makedirs(d, exist_ok=True)
    for item in os.listdir(s):
        ss = os.path.join(s, item)
        dd = os.path.join(d, item)
        if os.path.isdir(ss):
            _copytree(ss, dd)
        else:
            shutil.copy2(ss, dd)

_copytree(src, PROJECT)
shutil.rmtree('/tmp/extracted', ignore_errors=True)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

required = ['app.py','inference','visualization','segmentation','training','report']
missing  = [r for r in required if not os.path.exists(f'{PROJECT}/{r}')]
if missing:
    print(f"MISSING: {missing}  <- re-upload the zip")
else:
    print("All files present!")
    os.system('ls /content/BrainTumorAI/')

## Step 4 — Install Packages

Installs timm, albumentations, opencv, scipy.


In [ ]:
import subprocess, sys, os

pkgs = [
    'timm==0.9.12',
    'albumentations==1.3.1',
    'opencv-python-headless',
    'reportlab',
    'scipy',
]
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f"{'OK  ' if r.returncode==0 else 'FAIL'} {pkg}")

import importlib
for mod in ['timm', 'albumentations', 'cv2', 'scipy']:
    try:
        importlib.import_module(mod)
        print(f"  import {mod} OK")
    except Exception as e:
        print(f"  import {mod} FAILED: {e}")

print("\nStep 4 complete — continue to Step 5")


## Step 5 â€” Upload Dataset

**Option A** (cell below): Upload `USE-Me Test.zip` from your PC
**Option B** (second cell): Copy from Google Drive if already there

Dataset structure must be:
```
USE-Me Test/
  glioma/
  meningioma/
  notumor/
  pituitary/
```

In [ ]:
# OPTION A: Upload from PC
# Right-click "USE-Me Test" folder on PC -> Send to -> Compressed
from google.colab import files
import zipfile, os

print("Select 'USE-Me Test.zip' when picker opens...")
uploaded = files.upload()
for fname in uploaded:
    print(f"Extracting: {fname}")
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('/content/BrainTumorAI/')
    os.remove(fname)

# Optionally upload checkpoints too
print("\nSelect 'checkpoints.zip' if you have one (Cancel to skip)...")
try:
    uploaded2 = files.upload()
    for fname in uploaded2:
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/BrainTumorAI/')
        os.remove(fname)
except Exception:
    print("Checkpoints upload skipped")

os.system('ls /content/BrainTumorAI/')

In [ ]:
# OPTION B: Copy from Google Drive
# Use if dataset is at: MyDrive/NeuroScan_UNet/USE-Me Test/
import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/NeuroScan_UNet'
for name, dst in [
    ('USE-Me Test', '/content/BrainTumorAI/USE-Me Test'),
    ('checkpoints',  '/content/BrainTumorAI/checkpoints'),
]:
    src = f'{DRIVE_DATA}/{name}'
    if not os.path.exists(src):
        print(f"Not in Drive: {src}")
    elif os.path.exists(dst):
        print(f"Already exists: {dst}")
    else:
        shutil.copytree(src, dst)
        print(f"Copied: {dst}")

In [ ]:
# Verify dataset
from pathlib import Path
root  = Path('/content/BrainTumorAI/USE-Me Test')
total = 0
for cls in ['glioma', 'meningioma', 'notumor', 'pituitary']:
    p = root / cls
    n = len(list(p.glob('*.*'))) if p.exists() else 0
    flag = 'OK' if n > 0 else 'MISSING'
    print(f"  {cls:12s}: {n:4d} images  {flag}")
    total += n
print(f"\n  Total: {total} images")
assert total > 0, "No images found! Run Option A or B above first."

## Step 6 — Generate Classical Pseudo Masks (NO GradCAM)

Uses pure classical image processing to find hyperintense tumor regions.

**Why no GradCAM?** GradCAM uses classifier gradients — it finds what changes the
classification score most, not the actual tumour location. It can point to the wrong
hemisphere. Classical thresholding is physically correct: tumours are **bright** on MRI.

| Class | Method |
|-------|--------|
| glioma | Top 8–20% brightness, fill ring hole → solid disk |
| meningioma | Top 16–28% brightness, merge adjacent lobes |
| pituitary | Top 4–14% brightness, anatomical position gate |
| notumor | All-zero mask immediately |


In [ ]:
import sys, os, cv2
import numpy as np
from pathlib import Path

sys.path.insert(0, '/content/BrainTumorAI')
os.chdir('/content/BrainTumorAI')

from refinement.classical_mask_generator import generate_mask

USE_ME_TEST     = '/content/BrainTumorAI/USE-Me Test'
PSEUDO_MASK_DIR = '/content/BrainTumorAI/data/pseudo_masks'
os.makedirs(PSEUDO_MASK_DIR, exist_ok=True)

CLASSES = ['glioma', 'meningioma', 'pituitary', 'notumor']

count = skipped = 0
for cls in CLASSES:
    cls_dir = Path(USE_ME_TEST) / cls
    if not cls_dir.exists():
        print(f"  SKIP {cls} (not found)")
        continue
    images = sorted(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))
    for img_path in images:
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            skipped += 1
            continue
        mask = generate_mask(bgr, cls)   # uint8 {0,1}
        # Save as {stem}.png — BrainSegDataset finds by stem (flat folder)
        save_path = Path(PSEUDO_MASK_DIR) / (img_path.stem + '.png')
        cv2.imwrite(str(save_path), (mask * 255).astype(np.uint8))
        count += 1
    print(f"  {cls:12s}: {len(images)} images")

print(f"\nDone: {count} classical masks -> {PSEUDO_MASK_DIR}")


In [ ]:
# Backup masks to Google Drive
import shutil, os

DRIVE_DIR   = '/content/drive/MyDrive/NeuroScan_UNet'
drive_masks = f'{DRIVE_DIR}/pseudo_masks'
if os.path.exists(drive_masks):
    shutil.rmtree(drive_masks)
shutil.copytree(PSEUDO_MASK_DIR, drive_masks)
print(f"Backed up to Drive: {drive_masks}")


In [ ]:
from pathlib import Path
import cv2, numpy as np

mask_dir = Path(PSEUDO_MASK_DIR)
print(f"Masks in {mask_dir}:")
for cls in ['glioma', 'meningioma', 'pituitary', 'notumor']:
    masks = [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
             for f in sorted(mask_dir.glob('*.png'))
             if cls[:4].lower() in f.stem.lower()]
    if not masks:
        print(f"  {cls:12s}: no masks found")
        continue
    areas   = [100.0 * (m > 127).sum() / m.size for m in masks if m is not None]
    nonzero = sum(1 for a in areas if a > 0)
    print(f"  {cls:12s}: {len(areas):3d} masks  "
          f"nonzero={nonzero:3d}  "
          f"mean={np.mean(areas):.1f}%  "
          f"min={min(areas):.1f}%  max={max(areas):.1f}%")


In [ ]:
import matplotlib.pyplot as plt
import cv2, numpy as np
from pathlib import Path

USE_ME_TEST     = '/content/BrainTumorAI/USE-Me Test'
PSEUDO_MASK_DIR = '/content/BrainTumorAI/data/pseudo_masks'
CLASSES         = ['glioma', 'meningioma', 'pituitary', 'notumor']

fig, axes = plt.subplots(4, 4, figsize=(16, 16), facecolor='#0f172a')
for row, cls in enumerate(CLASSES):
    cls_dir  = Path(USE_ME_TEST) / cls
    mask_dir = Path(PSEUDO_MASK_DIR)
    images   = sorted(list(cls_dir.glob('*.jpg')) + list(cls_dir.glob('*.png')))[:2]
    for col_pair, img_path in enumerate(images):
        img_col  = col_pair * 2
        mask_col = col_pair * 2 + 1
        bgr = cv2.imread(str(img_path))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        axes[row][img_col].imshow(rgb)
        axes[row][img_col].set_title(f'{cls} orig', color='#00d4ff', fontsize=8)
        axes[row][img_col].axis('off')
        m_path = mask_dir / (img_path.stem + '.png')
        m = cv2.imread(str(m_path), cv2.IMREAD_GRAYSCALE) if m_path.exists()             else np.zeros((10, 10), dtype=np.uint8)
        axes[row][mask_col].imshow(m, cmap='gray', vmin=0, vmax=255)
        tc = '#10b981' if m is not None and m.any() else '#ef4444'
        axes[row][mask_col].set_title(
            f"mask {100*m.mean()/255:.1f}%", color=tc, fontsize=8)
        axes[row][mask_col].axis('off')

plt.suptitle('Classical Masks Preview (orig | mask)', color='white', fontsize=12)
plt.tight_layout()
DRIVE_DIR = '/content/drive/MyDrive/NeuroScan_UNet'
plt.savefig(f'{DRIVE_DIR}/classical_mask_preview.png', dpi=80,
            bbox_inches='tight', facecolor='#0f172a')
plt.show()
print("Preview saved to Drive")


## Step 7 — Train Attention U-Net on GPU  [LITE — ~8 min on T4]

Trains on **classical pseudo-masks** (no GradCAM, no classifier needed).

| Setting | Value | Why |
|---------|-------|-----|
| architecture | attention_unet | Attention gates — precise boundaries |
| base_filters | **16** | 2.0 M params — 4× lighter than full |
| loss | CombinedSegLoss | 0.5×Tversky + 0.3×Dice + 0.2×Boundary |
| tversky_alpha | 0.7 | Penalises over-segmentation (FP) |
| batch_size | **8** + grad_accum 2 | Effective batch = 16 |
| epochs | **40** | Attention U-Net converges fast |
| AMP | on | Mixed precision — ~2× speed |

Checkpoint saved as `best_unet.pth` — drop directly into `checkpoints/unet/` locally.


In [ ]:
import os

CONFIG = {
    'images_dir'     : '/content/BrainTumorAI/USE-Me Test',
    'pseudo_cache'   : '/content/BrainTumorAI/data/pseudo_masks',
    'image_size'     : 224,
    'in_channels'    : 3,
    'val_split'      : 0.15,
    'architecture'   : 'attention_unet',
    'base_filters'   : 16,
    'bilinear'       : True,
    'epochs'         : 40,
    'batch_size'     : 8,
    'grad_accum'     : 2,
    'lr'             : 3e-4,
    'tversky_alpha'  : 0.7,
    'tversky_beta'   : 0.3,
    'boundary_weight': 2.0,
    'threshold'      : 0.45,
    'num_workers'    : 2,
    'seed'           : 42,
    'out_dir'        : '/content/BrainTumorAI/checkpoints/unet',
    'vis_every'      : 10,
    'device'         : 'cuda',
}
os.makedirs(CONFIG['out_dir'], exist_ok=True)
for k, v in CONFIG.items():
    print(f"  {k:18s}: {v}")


In [ ]:
import sys, time, warnings, os
import numpy as np
import torch
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/BrainTumorAI')

from segmentation.attention_unet import AttentionUNet
from segmentation.unet           import UNet
from segmentation.losses         import CombinedSegLoss
from segmentation.metrics        import compute_all_metrics
from segmentation.dataset        import BrainSegDataset

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
DEVICE = torch.device(CONFIG['device'])

# ── Dataset ───────────────────────────────────────────────────────────────
def make_ds(train):
    return BrainSegDataset(
        image_dir    = CONFIG['images_dir'],
        pseudo_cache = CONFIG['pseudo_cache'],
        image_size   = CONFIG['image_size'],
        train        = train,
    )

full_ds = make_ds(train=True)
n_total = len(full_ds)
n_val   = max(1, int(n_total * CONFIG['val_split']))
n_train = n_total - n_val

gen = torch.Generator().manual_seed(CONFIG['seed'])
tr_sub, va_sub = random_split(full_ds, [n_train, n_val], generator=gen)
va_sub.dataset = make_ds(train=False)

ldr_kw = dict(batch_size=CONFIG['batch_size'],
              num_workers=CONFIG['num_workers'], pin_memory=True)
train_loader = DataLoader(tr_sub, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(va_sub, shuffle=False, **ldr_kw)
print(f"Train: {n_train}  |  Val: {n_val}  |  Total: {n_total}")

# ── Model ─────────────────────────────────────────────────────────────────
arch = CONFIG['architecture'].lower()
if 'attention' in arch:
    model = AttentionUNet(
        in_channels  = CONFIG['in_channels'],
        base_filters = CONFIG['base_filters'],
        bilinear     = CONFIG['bilinear'],
    ).to(DEVICE)
    print(f"Attention U-Net  params: {model.count_parameters():,}")
else:
    model = UNet(
        in_channels  = CONFIG['in_channels'],
        base_filters = CONFIG['base_filters'],
        bilinear     = CONFIG['bilinear'],
    ).to(DEVICE)
    print(f"U-Net params: {model.count_parameters():,}")

# ── Loss + Optimiser + Scheduler + AMP ───────────────────────────────────
criterion = CombinedSegLoss(
    tversky_alpha   = CONFIG['tversky_alpha'],
    tversky_beta    = CONFIG['tversky_beta'],
    boundary_weight = CONFIG['boundary_weight'],
    w_tversky  = 0.5,
    w_dice     = 0.3,
    w_boundary = 0.2,
)
optimizer = torch.optim.Adam(
    model.parameters(), lr=CONFIG['lr'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-7)
scaler = GradScaler()

print(f"Loss      : CombinedSegLoss (alpha={CONFIG['tversky_alpha']}, beta={CONFIG['tversky_beta']})")
print(f"Scheduler : CosineAnnealingWarmRestarts(T_0=10, T_mult=2)")
print(f"Grad accum: {CONFIG['grad_accum']} steps  (effective batch = {CONFIG['batch_size']*CONFIG['grad_accum']})")
print(f"AMP       : enabled")
print()
print(f"{'Ep':>4} | {'TrLoss':>8} {'TrDice':>7} {'TrIoU':>6} | "
      f"{'VaLoss':>8} {'VaDice':>7} {'VaIoU':>6} | {'LR':>9} | {'s':>5}")
print("-" * 85)

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    tots  = {'loss': 0., 'dice': 0., 'iou': 0.}
    n     = 0
    accum = CONFIG['grad_accum']

    for step, batch in enumerate(loader):
        imgs  = batch['image'].to(DEVICE)
        masks = batch['mask'].to(DEVICE)

        if train:
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, masks)   # scalar — no tuple unpacking
            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or (step + 1) == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
        else:
            with torch.no_grad(), autocast():
                logits = model(imgs)
                loss   = criterion(logits, masks)

        bs  = imgs.size(0)
        met = compute_all_metrics(logits.detach().float(), masks, CONFIG['threshold'])
        tots['loss'] += loss.item() * bs
        tots['dice'] += met['dice']  * bs
        tots['iou']  += met['iou']   * bs
        n += bs

    return {k: v / max(n, 1) for k, v in tots.items()}


history   = {'tr_loss': [], 'tr_dice': [], 'tr_iou': [],
             'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice = 0.
best_ckpt = None
OUT_DIR   = CONFIG['out_dir']
optimizer.zero_grad()

print(f"{'Ep':>4} | {'TrLoss':>8} {'TrDice':>7} {'TrIoU':>6} | "
      f"{'VaLoss':>8} {'VaDice':>7} {'VaIoU':>6} | {'LR':>9} | {'s':>5}")
print("-" * 85)

for epoch in range(1, CONFIG['epochs'] + 1):
    t0  = time.time()
    tr  = run_epoch(train_loader, True)
    val = run_epoch(val_loader,   False)
    scheduler.step(epoch)
    lr_now = optimizer.param_groups[0]['lr']

    for k in ['loss', 'dice', 'iou']:
        history[f'tr_{k}'].append(tr[k])
        history[f'val_{k}'].append(val[k])

    print(f"{epoch:>4d} | {tr['loss']:>8.4f} {tr['dice']:>7.4f} {tr['iou']:>6.4f} | "
          f"{val['loss']:>8.4f} {val['dice']:>7.4f} {val['iou']:>6.4f} | "
          f"{lr_now:>9.2e} | {time.time()-t0:>5.1f}s", flush=True)

    if val['dice'] > best_dice:
        best_dice = val['dice']
        ckpt_path = f"{OUT_DIR}/best_unet.pth"
        torch.save({
            'epoch'              : epoch,
            'model_state_dict'   : model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_dice'           : best_dice,
            'val_iou'            : val['iou'],
            'config': {
                'architecture': CONFIG['architecture'],
                'in_channels' : CONFIG['in_channels'],
                'base_filters': CONFIG['base_filters'],
                'bilinear'    : CONFIG['bilinear'],
                'image_size'  : CONFIG['image_size'],
                'threshold'   : 0.45,
            },
        }, ckpt_path)
        best_ckpt = ckpt_path
        print(f"         *** New best: ep={epoch}  dice={best_dice:.4f}  -> best_unet.pth")

    if CONFIG['vis_every'] > 0 and (epoch % CONFIG['vis_every'] == 0 or epoch == 1):
        model.eval()
        fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor='#0f172a')
        mean_np = np.array([0.485, 0.456, 0.406])
        std_np  = np.array([0.229, 0.224, 0.225])
        with torch.no_grad():
            for batch in val_loader:
                imgs2 = batch['image'].to(DEVICE)
                probs = torch.sigmoid(model(imgs2).float()).cpu()
                for i in range(min(4, imgs2.size(0))):
                    t = imgs2[i].cpu().numpy()
                    if t.shape[0] == 1:
                        t = np.repeat(t, 3, axis=0)
                    axes[0][i].imshow((t.transpose(1,2,0)*std_np+mean_np).clip(0,1))
                    axes[0][i].set_title(f'ep{epoch}', color='#00d4ff', fontsize=8)
                    axes[0][i].axis('off')
                    axes[1][i].imshow(probs[i,0].numpy(), cmap='RdYlGn', vmin=0, vmax=1)
                    axes[1][i].set_title(f"dice={val['dice']:.3f}", color='#10b981', fontsize=8)
                    axes[1][i].axis('off')
                break
        plt.tight_layout()
        plt.savefig(f"{OUT_DIR}/vis_ep{epoch:03d}.png", dpi=80,
                    bbox_inches='tight', facecolor='#0f172a')
        plt.show()
        plt.close()

print("=" * 85)
print(f"  Training complete!  Best Val Dice: {best_dice:.4f}")
print(f"  Checkpoint: {best_ckpt}")
print("=" * 85)


## Step 8 â€” Save to Drive + Download

In [ ]:
import matplotlib.pyplot as plt, os

# Re-declare in case of variable loss after restart
DRIVE_DIR = '/content/drive/MyDrive/NeuroScan_UNet'
OUT_DIR   = CONFIG['out_dir']

fig, axes = plt.subplots(1, 3, figsize=(15,4), facecolor='#0f172a')
for ax, (key, color) in zip(axes, zip(
        ['loss','dice','iou'], ['#ef4444','#10b981','#3b82f6'])):
    ax.plot(history[f'tr_{key}'],  color=color, lw=2, label='Train')
    ax.plot(history[f'val_{key}'], color=color, lw=2, ls='--', alpha=0.7, label='Val')
    ax.set_title(key.upper(), color='white', fontsize=12)
    ax.set_facecolor('#0f172a'); ax.tick_params(colors='#94a3b8')
    ax.legend(facecolor='#1e293b', labelcolor='white', fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('#334155')
plt.suptitle(f'U-Net Training -- Best Val Dice: {best_dice:.4f}',
             color='white', fontsize=13)
plt.tight_layout()
curves_path = f'{OUT_DIR}/training_curves.png'
plt.savefig(curves_path, dpi=120, bbox_inches='tight', facecolor='#0f172a')
plt.show()
print(f'Curves saved: {curves_path}')


In [ ]:
import shutil, glob, os

DRIVE_DIR  = '/content/drive/MyDrive/NeuroScan_UNet'
DRIVE_UNET = f'{DRIVE_DIR}/checkpoints_unet'
OUT_DIR    = CONFIG['out_dir']
os.makedirs(DRIVE_UNET, exist_ok=True)

if best_ckpt and os.path.exists(best_ckpt):
    shutil.copy2(best_ckpt, DRIVE_UNET)
    print(f'Checkpoint -> Drive: {os.path.basename(best_ckpt)}')
else:
    print('WARNING: best_ckpt not found!')

if os.path.exists(curves_path):
    shutil.copy2(curves_path, DRIVE_UNET)
    print('Training curves -> Drive')

vis_files = glob.glob(f'{OUT_DIR}/vis_ep*.png')
for vf in vis_files:
    shutil.copy2(vf, DRIVE_UNET)
print(f'{len(vis_files)} visualisation images -> Drive')

print(f'\nAll saved to: {DRIVE_UNET}')
print('(Persists after Colab disconnects)')
print('\nFiles in Drive:')
for f in sorted(glob.glob(f'{DRIVE_UNET}/*')):
    print(f'  {os.path.basename(f):50s} {os.path.getsize(f)//1024:5d} KB')


In [ ]:
from google.colab import files
import os

if best_ckpt and os.path.exists(best_ckpt):
    print(f'Downloading: {os.path.basename(best_ckpt)}')
    print('Download dialog will appear in your browser...')
    files.download(best_ckpt)
    print('\nDone! Place the .pt file in:')
    print('  checkpoints/unet/   on your local PC')
else:
    # Fallback: download from Drive
    DRIVE_UNET = '/content/drive/MyDrive/NeuroScan_UNet/checkpoints_unet'
    pts = sorted(__import__('glob').glob(f'{DRIVE_UNET}/unet*.pt'))
    if pts:
        print(f'Downloading from Drive: {os.path.basename(pts[-1])}')
        files.download(pts[-1])
    else:
        print('ERROR: No checkpoint found. Check Drive or re-run training.')


## Step 9 â€” Deploy Locally

```
1. Move downloaded .pt to:
   C:\Users\HP\Desktop\PRO(B)\BrainTumorAI\checkpoints\unet\

2. Run:
   cd "C:\Users\HP\Desktop\PRO(B)\BrainTumorAI"
   python app.py

3. Open http://localhost:7862
   Upload MRI â†’ expand "Tumour Segmentation (U-Net)"
   Badge shows: ðŸŸ¢ Live U-Net
```

In [ ]:
import glob, os, torch
print("="*60)
print("  TRAINING COMPLETE")
print("="*60)
print(f"  Best Val Dice : {best_dice:.4f}")
print(f"  Checkpoint    : {os.path.basename(best_ckpt)}")
print(f"  Epochs        : {CONFIG['epochs']}")
print(f"  Image size    : {CONFIG['image_size']}x{CONFIG['image_size']}")
print(f"  Model params  : {model.count_parameters():,}")
print(f"  GPU           : {torch.cuda.get_device_name(0)}")
print()
for f in sorted(glob.glob(f'{DRIVE_UNET}/*')):
    print(f"  {os.path.basename(f):45s} {os.path.getsize(f)//1024:5d} KB")
print("="*60)